# Interactive Sample TOF Plot
This notebook mirrors the plotting logic from `plot_sample_tof.py` and adds a single control to choose the input data path.

In [5]:
from typing import Literal
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import display, clear_output


from joint_tof_opt.plotting import load_plot_config
from joint_tof_opt.tof_batch_process import compute_tof_data_series


def load_gen_config():
    config_path = Path("../experiments/tof_config.yaml")
    if config_path.exists():
        with open(config_path, "r") as f:
            return yaml.safe_load(f)
    raise FileNotFoundError("Could not find experiments/tof_config.yaml")


def plot_sample_tof(
    ppath: Path,
    plot_type: Literal["distribution", "density"],
    show_10pct_max_line: bool = False,
):
    """
    Plot a sample time-of-flight histogram.

    Parameters:
    -----------
    ppath : str or Path
        Path to the .npz to the partial path table
    plot_type : str
        Type of plot: 'distribution' or 'density'
    show_10pct_max_line : bool
        If True, draw a horizontal line at 10% of the histogram maximum.
    """
    gen_config = load_gen_config()
    gen_config["datapoint_count"] = 10  # Only generate 10 ToFs for quick loading

    # Load data
    tof_data = compute_tof_data_series(ppath, gen_config, False, False)
    print(tof_data.var_series)

    # Get first row (first time point)
    tof_histogram = tof_data.tof_series.numpy()
    tof_histogram = np.mean(tof_histogram, axis=0)
    # Convert from seconds to nanoseconds
    bin_edges_ns = tof_data.bin_edges.numpy() * 1e9

    # Calculate bin centers for the line plot
    bin_centers = (bin_edges_ns[:-1] + bin_edges_ns[1:]) / 2

    # Calculate bin widths
    bin_widths = np.diff(bin_edges_ns)

    # Normalize to density if requested
    if plot_type == "density":
        # Density: histogram / (sum * bin_width)
        y_values = tof_histogram / (np.sum(tof_histogram) * bin_widths)
        ylabel = "Probability Density"
    else:
        # Distribution: raw counts
        y_values = tof_histogram
        ylabel = "Count"

    # Load plot configuration
    load_plot_config()

    # Apply rcParams
    load_plot_config()
    # Get first color from the color cycle
    bar_color = plt.rcParams["axes.prop_cycle"].by_key()["color"][0]

    # Create figure
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.grid(True)
    plt.gca().set_axisbelow(True)

    # Plot bars with black edge
    ax.bar(
        bin_centers,
        y_values,
        width=bin_widths,
        color=bar_color,
        edgecolor="black",
        linewidth=0.5,
        label="Histogram",
    )

    if show_10pct_max_line and np.size(y_values) > 0:
        max_value = float(np.max(y_values))
        if max_value > 0:
            ten_percent_max = 0.1 * max_value
            ax.axhline(
                y=ten_percent_max,
                color="crimson",
                linestyle="--",
                linewidth=1.2,
                label="10% of max",
            )

    # Labels and title
    ax.set_xlabel("Time of Flight (ns)")
    ax.set_ylabel(ylabel)
    ax.set_title("Distribution Time-of-Flight (DTOF)")
    ax.set_yscale("log")
    # ax.legend()

    plt.show()
    plt.close()


DATA_PATH_OPTIONS = [f"../data/experiment_{i:04d}.npz" for i in range(8)]

In [6]:
output = widgets.Output()


def run_interactive_plot(data_path: str, show_10pct_max_line: bool):
    with output:
        clear_output(wait=True)
        plot_sample_tof(
            Path(data_path),
            plot_type="distribution",
            show_10pct_max_line=show_10pct_max_line,
        )


data_selector = widgets.Dropdown(
    options=DATA_PATH_OPTIONS,
    value=DATA_PATH_OPTIONS[0],
    description="Data path:",
    layout=widgets.Layout(width="420px"),
)

show_10pct_line_checkbox = widgets.Checkbox(
    value=False,
    description="Show 10% of max line",
    indent=False,
)

widgets.interactive_output(
    run_interactive_plot,
    {
        "data_path": data_selector,
        "show_10pct_max_line": show_10pct_line_checkbox,
    },
)
display(widgets.VBox([data_selector, show_10pct_line_checkbox, output]))